# 📖 Notebook 3: Health Checks & Sticky Sessions

A load balancer is only as good as its view of the backends. If one of your
servers starts returning 500s but the LB keeps sending traffic to it, users
get a broken experience half the time.

In this notebook we'll cover:

1. **Liveness vs readiness** — two different questions the LB asks each
   backend.
2. A small simulator where a backend turns sick and the LB drains it from
   the pool.
3. **Sticky sessions (session affinity)** — pinning a user to one backend.
   Useful, but dangerous.

## Learning Objectives

By the end of this notebook, you'll understand:

- The difference between **liveness** ("are you alive?") and **readiness**
  ("are you ready to serve traffic right now?")
- How a health-check loop keeps the pool of backends up to date
- What sticky sessions are, when to use them, and why stateless backends are
  usually better


## 🛠️ Setup

From the lab folder:

```bash
cd 01-foundations/load-balancing
uv sync
```

### Kernel selection

In VS Code, click the kernel picker at the **top-right** of this notebook and
choose the `.venv` interpreter (it will be named something like
`.venv (Python 3.x)`).

If the `.venv` kernel doesn't appear in the list, reload the VS Code window:

- `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Windows/Linux)
- Type and select **"Developer: Reload Window"**

No Docker or external services are needed for this lab — everything runs
in-process with plain Python.


## 🏥 Liveness vs Readiness

People often say "health check" as one thing, but there are really **two**
questions we want to ask a backend:

- **Liveness**: *"Are you alive at all?"* If the answer is no, the process
  is hung or crashed and it should be **restarted** (in Kubernetes: killed
  and recreated).
- **Readiness**: *"Are you ready to take traffic right now?"* A backend can
  be alive but not ready — for example, it's still loading a huge ML model
  into memory, or it's warming up connection pools. In that case we don't
  want to **kill** it, we just want the **load balancer** to temporarily
  **stop sending it requests**.

Mixing these up causes real outages:

- A readiness check that doubles as a liveness check will **kill** backends
  during a normal warmup → cascading restarts.
- A liveness check that's too aggressive will restart a backend that's just
  briefly slow → thrashing.

Keep them separate, and keep them **cheap**.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
import random

class HealthStatus(BaseModel):
    """What a backend reports when the LB probes it."""
    alive: bool      # liveness: is the process responsive at all?
    ready: bool      # readiness: will it serve real traffic right now?


class Backend:
    """A fake backend with a health state we can tamper with for the demo."""

    def __init__(self, name: str):
        self.name = name
        self._alive = True
        self._ready = True
        self.handled = 0

    # --- things the LB calls ---
    def probe(self) -> HealthStatus:
        """Cheap endpoint the LB hits every few seconds."""
        return HealthStatus(alive=self._alive, ready=self._ready)

    def handle(self, request_id: int) -> str:
        if not self._alive:
            raise RuntimeError(f"{self.name} is DOWN and cannot serve")
        self.handled += 1
        return f"{self.name} handled req-{request_id}"

    # --- things our simulator calls to change the backend's state ---
    def crash(self):     self._alive = False; self._ready = False
    def recover(self):   self._alive = True;  self._ready = True
    def warmup(self):    self._alive = True;  self._ready = False  # alive but not ready
    def warmup_done(self): self._ready = True


Backend("demo").probe()


## 🔁 A tiny health-check loop

A real load balancer runs a background task that probes every backend on a
schedule. Each backend is in one of three LB-visible states:

- **IN_POOL** — receives traffic
- **DRAINING** — probe says not ready, stop sending new traffic
- **DEAD** — probe fails (or times out); same effect as draining, but we
  also log it louder

We'll implement that as a single function you can call step-by-step.


In [ ]:
class HealthAwareLB:
    """Round-robin load balancer that consults health checks."""

    def __init__(self, backends: list[Backend]):
        self.backends = backends
        # pool = currently eligible backends (derived from health checks)
        self.pool: list[Backend] = list(backends)
        self.i = 0

    def run_health_checks(self) -> None:
        """Re-evaluate the pool based on each backend's current probe."""
        new_pool = []
        for b in self.backends:
            status = b.probe()
            if not status.alive:
                print(f"  💀 {b.name} is DEAD (liveness failed) — removing from pool")
                continue
            if not status.ready:
                print(f"  🚧 {b.name} is not READY — draining from pool")
                continue
            new_pool.append(b)
        self.pool = new_pool
        print(f"  ➜ pool now has {len(self.pool)} backend(s): "
              f"{[b.name for b in self.pool]}")

    def route(self, request_id: int) -> str:
        if not self.pool:
            return "❌ no healthy backends — request fails"
        backend = self.pool[self.i % len(self.pool)]
        self.i += 1
        return backend.handle(request_id)


## 🎬 Scenario: a backend gets sick

We'll:

1. Start with 3 healthy backends.
2. Send a few requests — everyone shares the load.
3. `server-1` starts warming up (alive, not ready). The LB should drain it.
4. `server-2` crashes entirely. The LB should mark it dead.
5. `server-1` finishes warming up. The LB should add it back.


In [ ]:
backends = [Backend(f"server-{i}") for i in range(3)]
lb = HealthAwareLB(backends)

def tick(label):
    print(f"\n--- {label} ---")
    lb.run_health_checks()
    for req_id in range(6):
        print("  ", lb.route(req_id))

tick("Everyone healthy")

assert [b.name for b in lb.pool] == ["server-0", "server-1", "server-2"]
assert sum(b.handled for b in backends) == 6

In [ ]:
# server-1 starts warming up (e.g., reloading config)
backends[1].warmup()
tick("server-1 warming up")

# Readiness failing DRAINS the backend — it must not receive traffic...
assert backends[1] not in lb.pool
before = backends[1].handled
assert lb.route(999) and backends[1].handled == before, "a draining backend got traffic"
# ...but it is still alive, so nothing should have restarted it.
assert backends[1].probe().alive is True

In [ ]:
# server-2 crashes
backends[2].crash()
tick("server-2 crashed")

# A dead backend must never be handed a request — Backend.handle() would raise.
assert backends[2] not in lb.pool
assert [b.name for b in lb.pool] == ["server-0"]
served_before_crash = backends[2].handled   # frozen from here on

In [ ]:
# server-1 finishes warmup
backends[1].warmup_done()
tick("server-1 ready again (server-2 still dead)")

print("\nFinal handled counts:")
for b in backends:
    print(f"  {b.name}: {b.handled} requests")

# server-1 rejoins automatically once its readiness probe passes again.
assert [b.name for b in lb.pool] == ["server-0", "server-1"]
assert backends[2].handled == served_before_crash, "a dead backend was handed a request"

### What you should notice

- When `server-1` was warming up, the LB kept it **in the fleet** (didn't
  kill it) but stopped sending it traffic. That's readiness doing its job.
- When `server-2` crashed, the LB removed it and never attempted to send a
  request there — no user saw a 500.
- When `server-1` came back, it rejoined the rotation automatically.

This is the core magic of a production load balancer: **the set of backends
in the pool is not static**. It's the live answer to "which of you can
actually serve a request *right now*?"


## 🍪 Sticky sessions (session affinity)

By default, round-robin + a pool of backends assumes your backends are
**stateless** — any backend can serve any request. That's the dream: a
backend can die at any moment and nobody notices.

But what if a backend keeps some per-user state in local memory (a shopping
cart, a WebSocket connection, a file upload in progress)? Then every
request from that user needs to land on the **same** backend. That's a
**sticky session**.

The LB implements this by hashing a stable key from the request (usually a
cookie it set, or the client IP) and always routing that key to the same
backend.


In [ ]:
import hashlib

class StickyLB:
    """Sticky load balancer: a deterministic hash of session_id picks the backend.

    We use hashlib (md5) instead of Python's built-in hash() because hash() is
    randomized per process — the same session would land on different backends
    after a restart. md5 is fine here: we're not using it for security, just
    for a stable mapping.
    """

    def __init__(self, backends: list[Backend]):
        self.backends = backends

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode()).hexdigest(), 16)

    def route(self, session_id: str, request_id: int) -> str:
        idx = self._hash(session_id) % len(self.backends)
        return self.backends[idx].handle(request_id)


fresh = [Backend(f"server-{i}") for i in range(3)]
sticky = StickyLB(fresh)

# Three users, five requests each — each user should land on ONE backend,
# and that mapping is reproducible across runs and processes.
for user in ("alice", "bob", "carol"):
    print(f"\nUser {user}:")
    for r in range(5):
        print("  ", sticky.route(user, r))

# The promise of stickiness: one user -> one backend, every time, in every process.
for user in ("alice", "bob", "carol"):
    landed = {sticky.route(user, r).split()[0] for r in range(20)}
    assert len(landed) == 1, f"{user} was spread across {landed}"

# And it is a *stable* mapping, not just a consistent one within this run:
# md5 is deterministic, so these values are the same on any machine.
mapping = {u: sticky._hash(u) % 3 for u in ("alice", "bob", "carol")}
print("\nstable session -> backend index mapping:", mapping)
assert mapping == {"alice": 1, "bob": 2, "carol": 1}, mapping

# The catch that Notebook 4 fixes: change the fleet size and the mapping shatters.
reshuffled = {u: sticky._hash(u) % 4 for u in ("alice", "bob", "carol")}
print("same users after adding a 4th backend:", reshuffled)
moved = [u for u in mapping if mapping[u] != reshuffled[u]]
print(f"{len(moved)}/3 users moved just because N went from 3 to 4: {moved}")
assert len(moved) == 3, "every one of these three users lost their session"

### ✅ When sticky sessions help

- **In-memory session state** (a logged-in user's cart stored on the app
  server). Sending the next request to a different backend would lose the
  cart.
- **Long-lived connections** (WebSockets, HTTP/2 streams) — the connection
  physically lives on one backend.
- **Local caches** — if backend A already has this user's data warm in
  memory, sending them back to A is a free cache hit.

### ⚠️ Why they're usually a bad idea

1. **Uneven load.** If one user is a power user or a bot, their "sticky"
   backend gets slammed while others are idle.
2. **Lost state on failure.** If the sticky backend crashes, that user's
   session is gone. The whole point of a fleet was that any backend dying
   was harmless — stickiness undoes that.
3. **Harder deploys.** You can't just drain a backend; you have to migrate
   its users somewhere else first.
4. **Scaling pain.** Adding a new backend changes the hash ring, which can
   reshuffle many users at once.

### 🏆 The "best" approach: keep backends stateless

Move state **out** of the backend:

- Session data → **Redis** or a database
- File uploads → **S3** (or similar)
- WebSockets → a dedicated realtime layer (or use consistent hashing only
  for the WS tier)

Then any request can go to any backend, health checks + round-robin
"just work", and you sleep better at night.


## 🧠 Takeaways

- **Liveness** asks "are you alive?" — failure triggers a **restart**.
- **Readiness** asks "should I send traffic?" — failure drains the backend
  from the pool but keeps it running.
- A load balancer runs health checks on a loop; the pool of eligible
  backends is **dynamic**.
- **Sticky sessions** are useful but fragile. Prefer stateless backends with
  shared storage (Redis/DB) whenever you can.

🎉 That's the foundation of load balancing. From here you can dive into
real tools:

- **NGINX / HAProxy** — classic software load balancers.
- **Envoy** — what service meshes (Istio, Linkerd) are built on.
- **AWS ALB / GCP LB / Azure Front Door** — cloud-managed L7 load
  balancers.

All of them use the same ideas you just coded by hand.
